# Tutorial 5: multi-group design on the Smillie colon atlas

Notebooks 01 to 04 use a two-group dataset where the comparison is decided the
moment the groups exist. Real annotations are rarely that simple: the Smillie
colon single-cell atlas ships several nested cell-type levels plus an inflammation
status, so the interesting question is how to *compose* groups into an analysis.

This notebook covers the three composition steps the two-group workflow never
needs:

1. Sample Groups over a column with eight cell types, visualized with an
   **expression UMAP**.
2. A **Target built from several base groups** (`CD4+ T` and `CD8+ T`) against a
   single-group Baseline, with comparison-local display labels.
3. Base groups built from the **intersection of two metadata columns**
   (`Major_l3` and `Health`), which asks a condition question inside one cell type.

It is the same dataset and the same steps as the Smillie workflow in the Metabis
server tutorial, so the group counts here can be checked against the application.

## Before you run it

Install the repository with its development dependencies, then install the
example data:

```bash
python data/download_datasets.py
```

That script fetches the same author-maintained archive the Metabis server
deploys, so the notebook and the application read byte-identical inputs.

In [ ]:
# ruff: noqa: E402, I001
import os
import sys
import tempfile
import warnings
from pathlib import Path

from bokeh.io import output_notebook, show

os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "metabis-matplotlib"),
)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings(
    "ignore",
    message="n_jobs value 1 overridden to 1 by setting random_state.*",
)


def find_metabis_repo_root_for_import(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]
    for candidate in [*direct_candidates, *nested_candidates]:
        if (candidate / "src" / "metabis").exists():
            return candidate
    raise FileNotFoundError("Could not locate the Metabis repository root.")


repo_root_for_import = find_metabis_repo_root_for_import(Path.cwd().resolve())
src_path = repo_root_for_import / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from metabis import (
    AnalysisWorkspace,
    ExperimentAnalysis,
    ExpressionData,
    ExpressionPreprocessingConfig,
    GroupComparison,
    GroupComparisonSpec,
    MetabolicModel,
    ReactionData,
    ReactionPreprocessingConfig,
    SampleMetadata,
    build_reduction_presentation,
    plot_reduction_artifact,
    plot_statistical_histograms,
)
from metabis.analysis_workspace import RevisionKind
from metabis.version import __version__ as metabis_version

output_notebook()

In [ ]:
def find_metabis_repo_root(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]
    for candidate in [*direct_candidates, *nested_candidates]:
        if (
            (candidate / "pyproject.toml").exists()
            and (candidate / "demo").exists()
            and (candidate / "src" / "metabis").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the Metabis repository root.")


repo_root = find_metabis_repo_root(Path.cwd().resolve())
data_dir = repo_root / "data"
required_example_files = (
    "smillie_clusters_pool_reactions.tsv",
    "smillie_clusters_pool_expression_cpm.tsv",
    "smillie_metadata.tsv",
)
missing = [name for name in required_example_files if not (data_dir / name).is_file()]
if missing:
    raise FileNotFoundError(
        "Missing Smillie example files: "
        + ", ".join(missing)
        + ". Run `python data/download_datasets.py`."
    )

reaction_data_path = data_dir / "smillie_clusters_pool_reactions.tsv"
expression_data_path = data_dir / "smillie_clusters_pool_expression_cpm.tsv"
metadata_path = data_dir / "smillie_metadata.tsv"

# The atlas ships nested annotation levels. `Major_l4` is the cell-type level
# this notebook groups by; `Major_l3` and `Health` are intersected in section 5.
cell_type_column = "Major_l4"
target_group_ids = ("CD4+ T", "CD8+ T")
baseline_group_ids = ("Macrophages4",)

print(f"Metabis version: {metabis_version}")

## 1. Load the dataset the way the server loads it

The Smillie reaction matrix is Compass output computed on Recon2 identifiers, so
the model is aligned from `RECON2` before anything is loaded, exactly as in the
Th17 notebooks.

Two loading choices are specific to this dataset:

- The reaction matrix is a **dual** (`_pos`/`_neg`) Compass penalty matrix, so
  `compass_activity=True` decodes penalties into activity scores.
- The expression matrix is **already CPM-normalized**, so only `log1p` applies.
  Asking for `normalize_total` here would normalize an already-normalized matrix.

In [ ]:
model = MetabolicModel(use_cache=True)
model.apply_alignment_corrections(model.get_alignment_file_path("RECON2"))

reaction_data, reaction_validation = ReactionData.create_with_validation(
    str(reaction_data_path),
    data_mode="dual",
    metabolic_model=model,
    reaction_name_format="rxnRecon3DID",
    csv_separator="\t",
    samples_in="columns",
    check_old_identifiers=True,
    preprocessing=ReactionPreprocessingConfig(compass_activity=True),
)
expression_data, expression_validation = ExpressionData.create_with_validation(
    str(expression_data_path),
    gene_list=model.get_genes(),
    metabolic_model=model,
    gene_format="geneSymbols",
    preprocessing=ExpressionPreprocessingConfig(log1p=True),
    data_format="csv",
    csv_separator="\t",
    samples_in="columns",
)
assert reaction_validation["status"] == "success", reaction_validation["messages"]
assert expression_validation["status"] == "success", expression_validation["messages"]

reaction_df = reaction_data.get_reaction_df()
expression_df = expression_data.get_expression_df()
alignment_report = reaction_validation["alignment_report"]

print(
    f"Reaction matrix: {reaction_df.shape[0]} samples x "
    f"{reaction_df.shape[1]} reaction features"
)
print(
    f"Expression matrix: {expression_df.shape[0]} samples x "
    f"{expression_df.shape[1]} genes"
)
print(
    f"Aligned to Human1: {alignment_report['mapped_count']} reactions across "
    f"{alignment_report['subsystems_with_reactions']} subsystems"
)

The sample sheet is standalone metadata: its `cell` column carries the sample IDs
the matrices are keyed by, and every other column is available for grouping.

Smillie sample IDs are barcodes, so `display_label_column` points plot labels at
the readable `Cluster` annotation while the barcode remains the canonical
identity.

In [ ]:
metadata = SampleMetadata.from_table(
    metadata_path,
    sample_id_column="cell",
    display_label_column="Cluster",
)

dm = AnalysisWorkspace(
    metabolic_model=model,
    reaction_data=reaction_data,
    expression_data=expression_data,
)
dm.update_sample_metadata(metadata)

first_sample = reaction_data.get_sample_ids()[0]
print(f"Aligned samples: {len(reaction_data.get_sample_ids())}")
print(f"Metadata columns: {list(metadata.columns)}")
print(f"Display label for {first_sample}: {metadata.get_display_label(first_sample)}")

## 2. Sample Groups over eight cell types

`load_groups_from_metadata()` creates one atomic base group per distinct value of
the requested column. Nothing about this step decides a comparison: it only
declares the populations a later comparison may compose.

In [ ]:
grouping_result = dm.load_groups_from_metadata([cell_type_column])
group_ids = sorted(dm.get_group_ids())
sample_counts = {
    group_id: len(dm.get_group_samples(group_id)) for group_id in group_ids
}

for group_id, count in sample_counts.items():
    print(f"{group_id:>16}: {count:>4} samples")

assert sample_counts["CD4+ T"] == 200
assert sample_counts["CD8+ T"] == 100
assert sample_counts["Macrophages4"] == 100
print(f"\nUnassigned samples: {len(grouping_result.unassigned_samples)}")

## 3. Expression UMAP across all eight groups

With more than two populations, a single scatter of all groups is the fastest way
to see which cell types separate, which overlap, and where the annotation is
ragged. Both reductions are calculated before any artifact is installed, so the
workspace moves from "no reductions" to "both reductions" in one transaction.

UMAP is exploratory. Distances between clusters are not effect sizes, and nothing
here substitutes for the statistical comparison in section 4.

In [ ]:
analysis_dependencies = frozenset(
    {
        RevisionKind.DATASET,
        RevisionKind.METADATA,
        RevisionKind.GROUPS,
        RevisionKind.PREPROCESSING,
    }
)
analysis_snapshot = dm.capture_analysis_snapshot(analysis_dependencies)

analysis = ExperimentAnalysis(dm, auto_run=False)
analysis.calculate_umap(
    data_type="expression",
    n_neighbors=15,
    min_dist=0.1,
    n_variable_features=3000,
)
analysis.calculate_umap(
    data_type="reactions",
    n_neighbors=15,
    min_dist=0.1,
    n_variable_features=3000,
)

for result in analysis.get_reduction_results():
    analysis.install_reduction_artifact(
        result.build_artifact(
            provenance=analysis_snapshot.provenance,
            source_fingerprints=dict(dm.source_fingerprints),
        )
    )

dm.install_experiment_analysis(analysis_snapshot.provenance, analysis)
dm.refresh_experiment_analysis_presentation()
dm.set_analysis_group(target_group_ids[0])

print(
    "Installed reductions: "
    f"{[(a.key.method, a.key.data_type) for a in analysis.get_reduction_artifacts()]}"
)

In [ ]:
umap_expression_artifact = dm.get_reduction_artifact("UMAP", "expression")
assert umap_expression_artifact is not None
umap_expression_presentation = build_reduction_presentation(
    dm,
    umap_expression_artifact,
    width=900,
    height=650,
)
show(
    plot_reduction_artifact(
        umap_expression_artifact, umap_expression_presentation
    )
)

The same embedding in reaction space answers a different question: the expression
UMAP shows how the cell types differ transcriptionally, while the reaction UMAP
shows how much of that difference survives into predicted metabolic activity.
Cell types that separate in one view and not the other are the ones worth
comparing statistically.

In [ ]:
umap_reactions_artifact = dm.get_reduction_artifact("UMAP", "reactions")
assert umap_reactions_artifact is not None
umap_reactions_presentation = build_reduction_presentation(
    dm,
    umap_reactions_artifact,
    width=900,
    height=650,
)
show(plot_reduction_artifact(umap_reactions_artifact, umap_reactions_presentation))

## 4. A Target composed of several base groups

A comparison side is a *set* of base groups, not a single one. Here the Target is
the union of `CD4+ T` and `CD8+ T`, and the Baseline is `Macrophages4` alone.

`target_label` and `baseline_label` are comparison-local presentation only. They
rename the sides in plots and tables; they do not merge, rename, or replace the
base groups from section 2, and they are deliberately excluded from the
membership fingerprint.

Resolving membership *before* running any test is what makes the composition
auditable: the exact sample lists are frozen and can be checked for overlap.

In [ ]:
comparison_spec = GroupComparisonSpec.pairwise(
    target_group_ids,
    baseline_group_ids,
    target_label="CD4+CD8 T cells",
    baseline_label="Macrophages",
)
group_comparison = GroupComparison(
    spec=comparison_spec,
    test="wilcoxon",
    correction="fdr_bh",
)
membership = group_comparison.resolve_membership(dm)

overlap = set(membership.target_samples) & set(membership.baseline_samples)
print(
    f"Target   {membership.target_label}: {len(membership.target_samples)} samples "
    f"from {list(membership.target_base_group_ids)}"
)
print(
    f"Baseline {membership.baseline_label}: "
    f"{len(membership.baseline_samples)} samples "
    f"from {list(membership.baseline_base_group_ids)}"
)
print(f"Overlapping samples: {len(overlap)}")
print(f"Membership fingerprint: {membership.fingerprint}")

# 200 CD4+ T and 100 CD8+ T samples are deduplicated into one 300-sample Target.
assert len(membership.target_samples) == 300
assert len(membership.baseline_samples) == 100
assert not overlap

The orientation is fixed as Target minus Baseline, so positive values mean higher
in the combined T-cell Target.

In [ ]:
analysis_settings = {
    "pval_threshold": 0.05,
    "cohens_d_threshold": 0.5,
    "log2_fc_threshold": 0.0,
    "mean_difference_threshold": 0.0,
    "keep_reversed": True,
}

group_comparison.run_test(dm, for_genes=False)
group_comparison.run_test(dm, for_genes=True)

significant_reactions, _ = group_comparison.define_significant_elements(
    for_genes=False, **analysis_settings
)
significant_genes, _ = group_comparison.define_significant_elements(
    for_genes=True, **analysis_settings
)

comparison_snapshot = dm.capture_analysis_snapshot(analysis_dependencies)
comparison_artifact = group_comparison.build_artifact(
    provenance=comparison_snapshot.provenance,
    source_fingerprints=dict(dm.source_fingerprints),
    keep_reversed=analysis_settings["keep_reversed"],
)
dm.install_group_comparison_artifact(
    comparison_snapshot.provenance, comparison_artifact
)
dm.refresh_group_comparison_presentation()

print(
    f"{comparison_artifact.membership.target_label} minus "
    f"{comparison_artifact.membership.baseline_label}"
)
print(f"Significant reactions: {len(significant_reactions)}")
print(f"Significant genes: {len(significant_genes)}")

In [ ]:
plot_statistical_histograms(
    dm,
    parameters_to_plot=["pval", "cohens_d", "log2_fc"],
    return_plot=False,
    show_thresholds=True,
    for_genes=False,
)

## 5. Base groups from intersecting two metadata columns

The union in section 4 asks a between-cell-type question. To ask a *condition*
question inside one cell type, group by two columns at once: each base group then
becomes the intersection of one value from each, joined as `Major_l3 | Health`.

Regrouping is a workspace change, not a plot change. It advances the `groups`
revision, which is why the artifacts installed above are superseded and the new
comparison must capture a fresh snapshot rather than reuse the old provenance.

In [ ]:
groups_revision_before = analysis_snapshot.provenance.revisions.value_for(
    RevisionKind.GROUPS
)

dm.load_groups_from_metadata(["Major_l3", "Health"])
intersection_counts = {
    group_id: len(dm.get_group_samples(group_id))
    for group_id in sorted(dm.get_group_ids())
}

regrouped_snapshot = dm.capture_analysis_snapshot(analysis_dependencies)
groups_revision_after = regrouped_snapshot.provenance.revisions.value_for(
    RevisionKind.GROUPS
)

for group_id, count in intersection_counts.items():
    print(f"{group_id:>28}: {count:>4} samples")
print(f"\nGroups revision: {groups_revision_before} -> {groups_revision_after}")
assert groups_revision_after != groups_revision_before

`B cells | Inflamed` against `B cells | Non-inflamed` holds the cell type fixed
and varies inflammation status, which is the comparison the cell-type-only
grouping could not express.

In [ ]:
b_cell_comparison = GroupComparison(
    spec=GroupComparisonSpec.pairwise(
        ("B cells | Inflamed",),
        ("B cells | Non-inflamed",),
        target_label="Inflamed B cells",
        baseline_label="Non-inflamed B cells",
    ),
    test="wilcoxon",
    correction="fdr_bh",
)
b_cell_membership = b_cell_comparison.resolve_membership(dm)
assert len(b_cell_membership.target_samples) == 100
assert len(b_cell_membership.baseline_samples) == 62

b_cell_comparison.run_test(dm, for_genes=False)
b_cell_comparison.run_test(dm, for_genes=True)
b_cell_significant_reactions, _ = b_cell_comparison.define_significant_elements(
    for_genes=False, **analysis_settings
)
b_cell_significant_genes, _ = b_cell_comparison.define_significant_elements(
    for_genes=True, **analysis_settings
)

b_cell_artifact = b_cell_comparison.build_artifact(
    provenance=regrouped_snapshot.provenance,
    source_fingerprints=dict(dm.source_fingerprints),
    keep_reversed=analysis_settings["keep_reversed"],
)
dm.install_group_comparison_artifact(
    regrouped_snapshot.provenance, b_cell_artifact
)
dm.refresh_group_comparison_presentation()

b_cell_reaction_results = b_cell_artifact.get_test_results(for_genes=False)
print(
    f"{b_cell_membership.target_label} "
    f"({len(b_cell_membership.target_samples)} samples) minus "
    f"{b_cell_membership.baseline_label} "
    f"({len(b_cell_membership.baseline_samples)} samples)"
)
print(f"Significant reactions: {len(b_cell_significant_reactions)}")
print(f"Significant genes: {len(b_cell_significant_genes)}")
print(
    "Smallest adjusted p-value: "
    f"{b_cell_reaction_results['adjusted_pval'].min():.3f}"
)

This comparison returns **no** significant reactions or genes under the same
thresholds that gave hundreds for the cell-type contrast, and the smallest
adjusted p-value never crosses 0.05. That is the expected shape of the result,
not a configuration error: inflammation shifts metabolism inside one cell type
far less than the difference between T cells and macrophages, and holding the
cell type fixed also costs samples.

Reusing one threshold set across both comparisons is what makes that readable. The
ranked table below is still the right way to inspect a null-looking comparison:
it shows how far the strongest candidates fall short rather than hiding them
behind an empty significant set.

In [ ]:
b_cell_reaction_results.sort_values("adjusted_pval")[
    ["adjusted_pval", "cohens_d", "log2_fc", "mean_difference", "subsystems"]
].head(10)

## Takeaways

Group definition and comparison design are separate decisions. The same eight
base groups supported a union Target, and a second grouping of the same samples
supported a within-cell-type condition comparison, without either analysis
reaching past its own resolved membership.

Three properties carried the workflow: base-group IDs are scientific identity
while comparison labels are presentation, membership is resolved and checked for
overlap before any test runs, and regrouping advances the workspace revision so a
stale provenance cannot be reused.

Notebook 04 renders subsystem and graph artifacts from whichever comparison is
installed, so either comparison built here can be carried straight into it.